# Inference Optimization & Model Serving - Start Here

Welcome to Phase 30 of the Zero-to-AI curriculum. This module covers how to make LLM inference fast, cheap, and scalable: the techniques that turn a trained model into a production-grade service.

**Duration:** 6-8 hours right now, with planned expansion

> Status: this phase is intentionally published as a work in progress.
> Today, the reliable starting point is `03_serving_with_vllm.ipynb`. The remaining planned notebooks show the intended roadmap, not completed coverage.

---

## Prerequisites

- Completion of `14-local-llms/`
- Completion of `04-token/`
- Basic understanding of PyTorch devices and CUDA memory

## How To Use This Phase Right Now

1. Treat this as an introduction to serving and optimization, not a complete mastery path.
2. Start with the available vLLM notebook and use it to learn batching, runtime behavior, and measurement.
3. Pair this phase with `09-mlops/` and `14-local-llms/` for a more complete serving picture.

## Why Inference Optimization Matters

A 7B parameter model in FP16 uses ~14 GB of VRAM just to load. Every token it generates requires reading those weights plus the growing KV cache. At scale, inference cost dominates - often 10–100× the cost of training.

This phase teaches the techniques that reduce latency and cost:

```
Technique                    Speedup      Memory Savings
──────────────────────────────────────────────────────────
KV Cache + PagedAttention    2–4×         30–50% less waste
INT4 Quantization (AWQ)      2–3×         ~75% model size reduction
Continuous Batching          5–10× throughput   -
Speculative Decoding         1.5–2.5×     minimal overhead
Prefix Caching               1.3–2×       reuse repeated prompts
```

## Learning Path

| # | Notebook | Topic | Status |
|---|---------|-------|--------|
| 01 | `01_kv_cache_paged_attention.ipynb` | Visualizing and managing the KV cache | Planned |
| 02 | `02_quantization_deep_dive.ipynb` | Quantizing models from FP16 to INT4 (AWQ) | Planned |
| 03 | `03_serving_with_vllm.ipynb` | vLLM-based serving and batching | ✅ Available |
| 04 | `04_speculative_decoding.ipynb` | Speeding up inference with draft models | Planned |

### Start with the available notebook:

**`03_serving_with_vllm.ipynb`** - Set up a vLLM server, understand continuous batching, and measure throughput.

### Key Concepts You'll Learn

- **KV Cache**: Why autoregressive generation is memory-bound, not compute-bound
- **PagedAttention**: How vLLM avoids memory fragmentation (like virtual memory for attention)
- **Quantization**: AWQ, GPTQ, EXL2, GGUF - tradeoffs between quality and speed
- **Continuous Batching**: Serving many requests simultaneously without waiting for the longest
- **Speculative Decoding**: Using a small fast model to draft tokens, verified by the large model
- **Serving Runtimes**: vLLM vs TensorRT-LLM vs SGLang vs TGI

## Key Metrics

When benchmarking inference, measure these:

| Metric | What It Measures | Target |
|--------|-----------------|--------|
| **TTFT** (Time to First Token) | Latency until first output token | < 500ms for chat when possible |
| **TPS** (Tokens per Second) | Decode speed | 30-100+ tokens/sec for many apps |
| **Throughput** | Total tokens/sec across concurrent requests | Maximize |
| **Memory** | Peak VRAM usage | Fit your GPU budget |
| **Cost per 1M tokens** | Cost per million tokens served | Minimize |

---

## API Provider and Model Speed Benchmarks (May 2026)

If you're serving through an API instead of self-hosting, model selection and provider selection are both optimization problems. The fastest model on paper is not always the fastest or cheapest production choice for your workload.

### Two Different Leaderboards Matter

1. **Fastest overall output models**: useful when you need raw decode throughput.
2. **Best speed-quality-cost tradeoffs**: useful when you still need a capable assistant, coder, or reasoning model.

### Fastest Notable Models Right Now

| Model | Why It Matters |
|-------|----------------|
| **Mercury 2** | Current raw output-speed leader in the live leaderboard snapshot |
| **Granite 3.3 8B** | Extremely fast smaller model |
| **Granite 4.0 H Small** | Fast and cheap enough to matter for routing layers |
| **gpt-oss-120B (low/high)** | Remarkably fast for a much larger model family |
| **Gemini 3.1 Flash-Lite Preview** | Very strong hosted throughput for lightweight API use |

### Practical Speed vs Quality Snapshot

| Model | Intelligence Index | Speed (tok/s) | Approx Price ($/1M) | Why You'd Choose It |
|-------|-------------------|---------------|---------------------|---------------------|
| **GPT-5.5 (xhigh)** | 60 | 79 | $4.35 | Best overall quality |
| **Claude Opus 4.7 (max)** | 57 | 50 | $4.10 | Elite coding and analysis |
| **Gemini 3.1 Pro Preview** | 57 | 134 | $1.74 | Frontier quality with better value |
| **Qwen3.7 Max** | 57 | 200 | $1.43 | Frontier-capable and unusually fast |
| **Kimi K2.6** | 54 | 68 | $0.70 | Top open-weight quality |
| **MiMo-V2.5-Pro** | 54 | 61 | $0.71 | Long-context open-weight quality |
| **Grok 4.3 (high)** | 53 | 118 | $0.64 | Fast frontier-adjacent option |
| **DeepSeek V4 Pro (Max)** | 52 | 47 | $0.18 | Strong quality at very low price |
| **DeepSeek V4 Flash (Max)** | 47 | 117 | $0.06 | Best value open deployment |
| **GPT-5.4 mini (xhigh)** | 49 | 176 | $0.65 | Fast, strong hosted reasoning |
| **gpt-oss-120B (high)** | 33 | 327 | $0.20 | Cheap, extremely fast large model |
| **gpt-oss-20B (high)** | 24 | 243 | $0.07 | Cheap routing / budget workloads |
| **Nova Micro** | 10 | 307 | $0.03 | Ultra-cheap small hosted layer |
| **Qwen3.5 0.8B** | 10-11 | 83+ | $0.01 | Cheapest current general-purpose option |

*Data changes live on Artificial Analysis; treat these as May 2026 directional defaults, not permanent constants.*

### Optimization Decision Tree

```
Are you optimizing for raw speed first?
├─ Yes -> Check Mercury 2, Granite 3.3 8B, Granite 4.0 H Small, gpt-oss-120B
│
└─ No -> Are you optimizing for quality-per-dollar?
    ├─ Frontier quality -> Gemini 3.1 Pro Preview or Qwen3.7 Max
    ├─ Open-weight quality -> Kimi K2.6 or MiMo-V2.5-Pro
    ├─ Open-weight value -> DeepSeek V4 Flash / DeepSeek V4 Pro
    ├─ Budget routing -> gpt-oss-20B, Nova Micro, Qwen3.5 0.8B
    └─ Local serving -> vLLM or SGLang on the best model your hardware can sustain
```

### Important May 2026 Lesson

Inference optimization is no longer just about kernels, batching, and quantization. It is also about **model routing** across tiers:

- a tiny cheap model for classification or guardrails
- a mid-tier fast model for most requests
- a frontier model only for hard cases

That routing pattern usually saves more money than micro-optimizing a single premium model.